In [1]:
import datetime
import re
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import signal, stats
import scipy.io as sio

%config InlineBackend.figure_format = 'svg'


# =========================================================================
# Direct 36-Hour Butterworth Low-Pass Filter with 18-Hour NaN Edges
# =========================================================================
def butter_36h_nan_edges(series, cutoff_hours=36, nan_hours=18):
    """Filters data directly using a 36-hour Butterworth low-pass filter

    and masks the first and last `nan_hours` with NaN values to remove edge
    effects.
    """
    # 1. Clean missing values via linear interpolation for continuous filtering
    y = series.interpolate(method="linear").bfill().ffill().to_numpy()
    N = len(y)

    # 2. Design 36-hour Butterworth low-pass filter (fs = 1 hr^-1, fc = 1/36 hr^-1)
    fs = 1.0
    fc = 1.0 / cutoff_hours
    w_cutoff = fc / (fs / 2.0)  # Normalized Nyquist frequency
    sos = signal.butter(4, w_cutoff, btype="lowpass", analog=False, output="sos")

    # 3. Apply zero-phase Butterworth filter directly
    y_filtered = signal.sosfiltfilt(sos, y)

    # 4. Mask the first and last N hours with NaN
    if N >= (2 * nan_hours):
        y_filtered[:nan_hours] = np.nan
        y_filtered[-nan_hours:] = np.nan

    return y_filtered


# --- Load Data ---
oceanDF = pd.read_csv("12_13_output_ocean_heatflux.csv")
riverDf = pd.read_csv("12_13_output_river_heatflux.csv")
midHeatFluxDF = pd.read_csv("mid_heatflux_2012-2013.csv")
mid1314DF = pd.read_csv("mid_heatflux_2013-2014.csv")
up1213DF = pd.read_csv("upriver1213_heatflux.csv")
up1314DF = pd.read_csv("upriver1314_heatflux.csv")

# Clean inputs
up1213DF["Qnet_clean"] = up1213DF["Qnet"].interpolate().bfill().ffill()
up1314DF["Qnet_clean"] = up1314DF["Qnet"].interpolate().bfill().ffill()

# Build combined dataframe using FULL date range first
combinedDF = pd.DataFrame()
combinedDF["Tm (UTC)"] = pd.to_datetime(oceanDF["Tm (UTC)"])
combinedDF["Qnet"] = (oceanDF["Qnet"] + riverDf["Qnet"]) / 2.0
combinedDF["Qnet_clean"] = combinedDF["Qnet"].interpolate().bfill().ffill()


# =========================================================================
# APPLY BUTTERWORTH FILTER DIRECTLY (18-Hour Start/End set to NaN)
# =========================================================================
combinedDF["lowpass"] = butter_36h_nan_edges(combinedDF["Qnet_clean"])
midHeatFluxDF["lowpass"] = butter_36h_nan_edges(midHeatFluxDF["Qnet"])
mid1314DF["lowpass"] = butter_36h_nan_edges(mid1314DF["Qnet"])
up1213DF["lowpass"] = butter_36h_nan_edges(up1213DF["Qnet_clean"])
up1314DF["lowpass"] = butter_36h_nan_edges(up1314DF["Qnet_clean"])


# =========================================================================
# TRUNCATE combinedDF AFTER FILTERING
# =========================================================================
cutoff_date = pd.Timestamp("2013-01-22 23:59:59")
combinedDF = combinedDF[combinedDF["Tm (UTC)"] <= cutoff_date].reset_index(
    drop=True
)


# --- Save Output Files ---
combinedDF.to_csv("12_13_substituted_heatflux.csv", index=False)
midHeatFluxDF.to_csv("mid_heatflux_2012-2013.csv", index=False)
mid1314DF.to_csv("mid_heatflux_2013-2014.csv", index=False)
up1213DF.to_csv("upriver1213_heatflux.csv", index=False)
up1314DF.to_csv("upriver1314_heatflux.csv", index=False)


# =========================================================================
# Plotting Verification
# =========================================================================
tempFigLabels = ["Qnet Average", "Lowpass"]
tempfig = px.line(
    x=up1213DF["Tm (UTC)"],
    y=[up1213DF["Qnet"], up1213DF["lowpass"]],
    color_discrete_sequence=["blue", "crimson"],
    title="Heat Flux",
)

for idx in range(len(tempFigLabels)):
    tempfig.data[idx].name = tempFigLabels[idx]
    tempfig.data[idx].hovertemplate = (
        f"variable={tempFigLabels[idx]}<br>x=%{{x}}<br>value=%{{y}}<extra></extra>"
    )
    tempfig.data[idx].legendgroup = tempFigLabels[idx]

tempfig.update_layout(
    title=dict(text="Heat Flux for 2012-2013", font=dict(size=25))
)
tempfig.update_xaxes(tickangle=30)
tempfig.update_xaxes(rangeslider_visible=True)
tempfig.update_xaxes(
    range=[pd.Timestamp("2012-12-09"), pd.Timestamp("2013-01-22")]
)
tempfig.update_layout(
    xaxis_title="Date", yaxis_title="Heat Flux (W/m2)", legend_title="Location"
)

tempfig.show()